In [ ]:
# RAG SGU - PyPDFLoader Toi Gian
print("Flow: config -> ingest -> chunk -> vector store -> query")

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

BASE_DIR = Path.cwd()
SRC_DIR = BASE_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rag_core import RAGConfig, RAGPipeline, configure_runtime_environment

configure_runtime_environment()
load_dotenv(BASE_DIR / ".env")

print(f"Workspace: {BASE_DIR}")
print(f"SRC path: {SRC_DIR}")

In [ ]:
config = RAGConfig.from_env(base_dir=BASE_DIR)
pipeline = RAGPipeline(config)

print("Configuration loaded:")
print(f"- PDF dir: {config.pdf_dir}")
print(f"- Vector store: {config.vector_store_dir}")
print(f"- Chunk size: {config.chunk_size}")
print(f"- Chunk overlap: {config.chunk_overlap}")
print(f"- Retrieval k: {config.retrieval_k}")

In [ ]:
# Build or load index in the next cells.
print("PDF co text-layer -> build index moi; PDF scan/image-only -> load index da co")

In [ ]:
build_result = None

try:
    build_result = pipeline.build_index()
    print("Index built successfully")
    print(f"- Loaded PDFs: {len(build_result.ingestion.loaded_pdf_files)}")
    print(f"- Extracted docs: {len(build_result.ingestion.documents)}")
    print(f"- Created chunks: {len(build_result.chunks)}")

    if build_result.ingestion.scanned_pdf_files:
        print("Warning: Some PDFs have no text-layer (skipped):")
        for file_name in build_result.ingestion.scanned_pdf_files:
            print(f"  - {file_name}")
except ValueError as exc:
    print("Could not build from PDFs:")
    print(exc)
    print("Trying to load existing FAISS index in next cell...")

## Build or Load Index
Neu build tu PDF that bai vi khong co text-layer, se load FAISS index da ton tai.

In [ ]:
if build_result is None:
    pipeline.load_index()
    print("Loaded existing index from disk")
else:
    print("Using freshly built in-memory index")

def ask(question: str):
    result = pipeline.query(question)
    print(f"Question: {question}")
    print(f"Answer: {result['answer']}")

    if result["sources"]:
        print("Sources:")
        for source in result["sources"]:
            print(f"  - {source}")
    else:
        print("Sources: (none)")

    return result

In [ ]:
demo_result = ask("Muc tieu dao tao cua nganh Cong nghe thong tin la gi?")

## Out-of-scope Query Test
Cell tiep theo dung de test cau hoi ngoai tai lieu (ky vong fallback).

In [ ]:
out_of_scope_result = ask("Thoi tiet hom nay o TP.HCM nhu the nao?")